**Import Libraries**

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import pearsonr

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8,6)

**Load Test Dataset**

In [ ]:
DATA = Path("../data/processed")

test_df = pd.read_csv(
    DATA/"test_set.csv"
)

TARGET = "VO2max"

X_test = test_df.drop(columns=[TARGET])

y_test = test_df[TARGET]

**Load Saved Feature Selectors**

In [ ]:
MODEL_DIR = Path("../models")

variance_selector = joblib.load(
    MODEL_DIR/"variance_selector.pkl"
)

rfe = joblib.load(
    MODEL_DIR/"rfe_selector.pkl"
)
X_test = variance_selector.transform(X_test)
X_test = rfe.transform(X_test)

**Load Qauntile Models**

In [ ]:
q25_model = joblib.load(
    MODEL_DIR/"lightgbm_q25.pkl"
)

q50_model = joblib.load(
    MODEL_DIR/"lightgbm_q50.pkl"
)

q75_model = joblib.load(
    MODEL_DIR/"lightgbm_q75.pkl"
)

**Predict**

In [ ]:
pred25 = q25_model.predict(X_test)

pred50 = q50_model.predict(X_test)

pred75 = q75_model.predict(X_test)

**Standard Regression Metrics**

In [ ]:
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred50
    )
)
mae = mean_absolute_error(
    y_test,
    pred50
)
r2 = r2_score(
    y_test,
    pred50
)
mape = np.mean(
    np.abs(
        (y_test-pred50)/y_test
    )
)*100
results = pd.DataFrame({

"Metric":[

"RMSE",

"MAE",

"MAPE",

"R2"

],

"Value":[

rmse,

mae,

mape,

r2

]

})

results
results.to_csv(
"../results/regression_metrics.csv",
index=False
)

**Pinball Loss**

In [ ]:
def pinball_loss(
        y,
        pred,
        alpha):

    error=y-pred

    return np.mean(

        np.maximum(

            alpha*error,

            (alpha-1)*error

        )
    )
pin25 = pinball_loss(
    y_test,
    pred25,
    0.25
)

pin50 = pinball_loss(
    y_test,
    pred50,
    0.50
)

pin75 = pinball_loss(
    y_test,
    pred75,
    0.75
)

**Coverage Probability**

In [ ]:
coverage = np.mean(

(y_test>=pred25)&

(y_test<=pred75)

)
print(

coverage

)

**Prediction With Interval**

In [ ]:
interval_width = (

pred75 -

pred25

)

In [ ]:
interval_width.describe()

In [ ]:
print(

interval_width.mean()

)

**Calibration Plot**

In [ ]:
absolute_error=np.abs(

y_test-pred50

)

plt.scatter(

interval_width,

absolute_error

)

plt.xlabel(

"Prediction Interval Width"

)

plt.ylabel(

"Absolute Error"

)

plt.show()

**Residual Analysis**

In [ ]:
residuals=y_test-pred50
sns.histplot(

residuals,

bins=30,

kde=True

)

plt.scatter(

pred50,

residuals

)

plt.axhline(

0,

color="red"

)

plt.xlabel(

"Prediction"

)

plt.ylabel(

"Residual"

)

from scipy import stats

stats.probplot(

residuals,

plot=plt

)

**Interval Visualisation**

In [ ]:
plot_df = pd.DataFrame({

"Actual":y_test.values,

"Median":pred50,

"Lower":pred25,

"Upper":pred75

}).reset_index(drop=True)
plt.figure(figsize=(15,6))

idx=np.arange(100)

plt.plot(

idx,

plot_df["Actual"][:100],

label="Actual"

)

plt.plot(

idx,

plot_df["Median"][:100],

label="Median"

)

plt.fill_between(

idx,

plot_df["Lower"][:100],

plot_df["Upper"][:100],

alpha=.3,

label="Prediction Interval"

)

plt.legend()
plt.title("VO₂max Prediction Intervals")
plt.show()

**Correlation**

In [ ]:
corr,p=pearsonr(

y_test,

pred50

)

print(corr)

**Error Distribution By Fitness Group**

In [ ]:
evaluation_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": pred50
})

evaluation_df["Absolute_Error"] = np.abs(
    evaluation_df["Actual"] - evaluation_df["Predicted"]
)

evaluation_df["Fitness_Group"] = pd.cut(
    evaluation_df["Actual"],
    bins=[0,30,45,100],
    labels=["Low","Moderate","High"]
)
sns.boxplot(
    data=evaluation_df,
    x="Fitness_Group",
    y="Absolute_Error"
)

plt.title("Absolute Error by Fitness Group")
plt.show()

**Save Predictions**

In [ ]:
predictions = pd.DataFrame({

"Actual":y_test,

"Q25":pred25,

"Median":pred50,

"Q75":pred75,

"Interval_Width":interval_width,

"Residual":residuals

})

predictions.to_csv(

"../results/test_predictions.csv",

index=False

)

**Final Evaluation Summary**

In [ ]:
summary = pd.DataFrame({

"Metric":[

"RMSE",

"MAE",

"MAPE",

"R2",

"Coverage",

"Mean Interval Width",

"Pinball25",

"Pinball50",

"Pinball75"

],

"Value":[

rmse,

mae,

mape,

r2,

coverage,

interval_width.mean(),

pin25,

pin50,

pin75

]

})

summary

In [ ]:
summary.to_csv(
    "../results/final_model_summary.csv",
    index=False
)